# `richdem-rs` Interactive Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/richdem-rs/blob/main/examples/richdem_rs_demo.ipynb)
[![PyPI](https://img.shields.io/pypi/v/richdem-rs)](https://pypi.org/project/richdem-rs/)
[![crates.io](https://img.shields.io/crates/v/richdem.svg)](https://crates.io/crates/richdem)

**`richdem-rs`** is a pure-Rust reimplementation of Richard Barnes's [RichDEM](https://github.com/r-barnes/richdem) powered by [`terrio`](https://github.com/opengeos/terrio) for GDAL-free geospatial raster I/O.

This notebook demonstrates:
1. Generating a synthetic DEM and reading/writing GeoTIFFs with zero GDAL dependencies
2. Depression filling (Priority-Flood+Epsilon) and breaching (Lindsay 2016)
3. Flow accumulation (`D8`, `Dinf`, `Quinn`)
4. Terrain attributes (`slope_degrees`, `aspect`, `curvature`)
5. Depression Hierarchy and Fill-Spill-Merge water routing

In [ ]:
%pip install -q richdem-rs matplotlib

In [ ]:
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import richdem_rs as rd
from richdem_rs import depression_hierarchy as dephier

print(f"richdem-rs version: {rd.__version__}")

## 1. Synthetic DEM & GDAL-Free GeoTIFF Round-Trip

In [ ]:
dem = rd.generate_perlin_terrain(256, seed=42)
dem.geotransform = [500_000.0, 10.0, 0.0, 4_500_000.0, 0.0, -10.0]
dem.projection = "EPSG:32613"

with tempfile.TemporaryDirectory() as tmpdir:
    tif_path = Path(tmpdir) / "synthetic_dem.tif"
    rd.SaveGDAL(str(tif_path), dem, compress=True)
    dem_loaded = rd.LoadGDAL(str(tif_path))
    max_diff = float(np.max(np.abs(np.asarray(dem) - np.asarray(dem_loaded))))
    print(f"Loaded {dem_loaded.shape} {dem_loaded.dtype} GeoTIFF (max |diff| = {max_diff:.2e})")

## 2. Depression Filling & Breaching

In [ ]:
filled = rd.FillDepressions(dem, epsilon=True)
breached = rd.BreachDepressions(dem)

fill_depth = np.asarray(filled) - np.asarray(dem)
breach_cut = np.asarray(dem) - np.asarray(breached)
print(f"Max fill depth: {fill_depth.max():.4f}, Max breach cut: {breach_cut.max():.4f}")

## 3. Flow Accumulation & Terrain Attributes

In [ ]:
accum_d8 = rd.FlowAccumulation(filled, method="D8")
accum_dinf = rd.FlowAccumulation(filled, method="Dinf")
slope_deg = rd.TerrainAttribute(filled, attrib="slope_degrees")
curvature = rd.TerrainAttribute(filled, attrib="curvature")

fig, axes = plt.subplots(2, 2, figsize=(10, 9))
im0 = axes[0, 0].imshow(dem, cmap="terrain")
axes[0, 0].set_title("Elevation (DEM)")
fig.colorbar(im0, ax=axes[0, 0], shrink=0.8)

im1 = axes[0, 1].imshow(np.log1p(np.maximum(accum_dinf, 0)), cmap="Blues")
axes[0, 1].set_title("D-infinity Flow Accumulation (log1p)")
fig.colorbar(im1, ax=axes[0, 1], shrink=0.8)

im2 = axes[1, 0].imshow(slope_deg, cmap="magma")
axes[1, 0].set_title("Slope (degrees)")
fig.colorbar(im2, ax=axes[1, 0], shrink=0.8)

im3 = axes[1, 1].imshow(fill_depth, cmap="viridis")
axes[1, 1].set_title("Depression Fill Depth")
fig.colorbar(im3, ax=axes[1, 1], shrink=0.8)

for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Depression Hierarchy & Fill-Spill-Merge

In [ ]:
basin_dem = rd.generate_perlin_terrain(128, seed=7)
basin_dem[0, :] = -1.0
basin_dem[-1, :] = -1.0
basin_dem[:, 0] = -1.0
basin_dem[:, -1] = -1.0

labels = rd.get_new_depression_hierarchy_labels(basin_dem.shape)
dh, flowdirs = rd.get_depression_hierarchy(basin_dem, labels)

wtd = rd.rdarray(np.full(basin_dem.shape, 0.05, dtype=np.float64), no_data=-9999.0)
rd.fill_spill_merge(basin_dem, labels, flowdirs, dh, wtd)

ocean_vol = dh[dephier.OCEAN].water_vol
standing_vol = float(np.sum(wtd))
print(f"Depressions: {len(dh) - 1}, Standing water: {standing_vol:.2f}, Ocean outflow: {ocean_vol:.2f}")

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(wtd, cmap="PuBu")
ax.set_title("Fill-Spill-Merge Standing Water Depth")
ax.axis("off")
fig.colorbar(im, ax=ax, shrink=0.85)
plt.tight_layout()
plt.show()